# Tarea 3 — PyTorch

**Nombre del integrante:** Richard Yrady

**Cédula del integrante:** 30715365

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torchvision import datasets, transforms

NOTEBOOK_NUMBER = 2
DATA_DIR = "data"  # gitignored, next to this notebook
SEED = 42

INPUT_SIZE = 784  # 28 x 28 pixels
DEFAULT_HIDDEN_LAYERS = 2
DEFAULT_ARCHITECTURE = "rectangular"  # "rectangular" or "convolutional" (funnel)
DEFAULT_NEURONS = 512
DEFAULT_CLASSIFICATION = "original"  # "original" (10 classes) or "simplified" (4 classes)

ORIGINAL_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
SIMPLIFIED_CLASSES = ["Top", "Bottom", "Footwear", "Bag"]

# Original label -> simplified label (0 = Top, 1 = Bottom, 2 = Footwear, 3 = Bag)
SIMPLIFIED_MAP = [0, 1, 0, 1, 0, 2, 0, 2, 3, 2]

# Optimizations that can be toggled from the menu
DEFAULT_OPTIMIZATIONS = {
    "optimizer": "adam",          # "adam" or "sgd" (SGD uses momentum 0.9)
    "learning_rate": 1e-3,
    "batch_size": 128,
    "weight_decay": 0.0,
    "dropout": 0.0,               # 0 disables dropout
    "batch_norm": False,
    "early_stopping": False,
    "patience": 3,                # epochs without validation improvement
    "validation_fraction": 0.1,   # share of training data used for early stopping
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 2. Datos (Fashion MNIST original y clases simplificadas)

In [ ]:
def load_split(train):
    """Downloads one Fashion MNIST split and returns it as tensors.

    Pixels are scaled to [0, 1], exactly what transforms.ToTensor() produces,
    and there is no Normalize outside the model.
    """
    dataset = datasets.FashionMNIST(DATA_DIR, train=train, download=True,
                                    transform=transforms.ToTensor())
    images = dataset.data.float().div(255.0).unsqueeze(1)  # (N, 1, 28, 28)
    labels = dataset.targets.clone()
    return images, labels


TRAIN_IMAGES, TRAIN_LABELS = load_split(train=True)
TEST_IMAGES, TEST_LABELS = load_split(train=False)
SIMPLIFIED_MAP_TENSOR = torch.tensor(SIMPLIFIED_MAP)


def class_names(classification):
    return ORIGINAL_CLASSES if classification == "original" else SIMPLIFIED_CLASSES


def get_data(classification, train):
    """Returns (images, labels) with labels in the requested classification."""
    images, labels = (TRAIN_IMAGES, TRAIN_LABELS) if train else (TEST_IMAGES, TEST_LABELS)
    if classification == "simplified":
        labels = SIMPLIFIED_MAP_TENSOR[labels]
    return images, labels


print(f"Train: {len(TRAIN_LABELS)} images | Test: {len(TEST_LABELS)} images")

## 3. Modelos (rectangular y convolucional)

In [ ]:
def output_size_for(classification):
    return len(class_names(classification))


def funnel_neurons(hidden_layers, input_size, output_size):
    """Linear interpolation between input and output: every layer shrinks by the same amount.

    neurons(i) = round(input + (output - input) * i / (L + 1)), i = 1..L
    With 2 hidden layers and 10 outputs: 784 -> 526 -> 268 -> 10.
    """
    step = (output_size - input_size) / (hidden_layers + 1)
    return [round(input_size + step * i) for i in range(1, hidden_layers + 1)]


def hidden_sizes_for(architecture, hidden_layers, neurons, output_size):
    if architecture == "convolutional":
        return funnel_neurons(hidden_layers, INPUT_SIZE, output_size)
    return [neurons] * hidden_layers


def build_model(hparams):
    """nn.Sequential: Flatten, then (Linear [BatchNorm1d] ReLU [Dropout]) per hidden layer, then Linear."""
    optimizations = hparams["optimizations"]
    layers = [nn.Flatten()]
    previous = hparams["input_size"]
    for size in hparams["neurons_per_layer"]:
        layers.append(nn.Linear(previous, size))
        if optimizations["batch_norm"]:
            layers.append(nn.BatchNorm1d(size))
        layers.append(nn.ReLU())
        if optimizations["dropout"] > 0:
            layers.append(nn.Dropout(optimizations["dropout"]))
        previous = size
    layers.append(nn.Linear(previous, hparams["output_size"]))
    return nn.Sequential(*layers)


def new_hyperparameters(classification=DEFAULT_CLASSIFICATION,
                        architecture=DEFAULT_ARCHITECTURE,
                        hidden_layers=DEFAULT_HIDDEN_LAYERS,
                        neurons=DEFAULT_NEURONS,
                        optimizations=None):
    output_size = output_size_for(classification)
    options = dict(DEFAULT_OPTIMIZATIONS)
    options.update(optimizations or {})
    return {
        "architecture": architecture,
        "classification": classification,
        "input_size": INPUT_SIZE,
        "output_size": output_size,
        "hidden_layers": hidden_layers,
        "neurons_per_layer": hidden_sizes_for(architecture, hidden_layers, neurons, output_size),
        "epochs": 0,
        "optimizations": options,
        "loss_history": [],
        "validation_loss_history": [],
    }


def describe(hparams):
    sizes = [hparams["input_size"]] + hparams["neurons_per_layer"] + [hparams["output_size"]]
    opts = hparams["optimizations"]
    print("=" * 56)
    print("  RED NEURONAL")
    print("=" * 56)
    print(f"  Tipo              : {hparams['architecture']}")
    print(f"  Clasificación     : {hparams['classification']} ({hparams['output_size']} clases)")
    print(f"  Capas ocultas     : {hparams['hidden_layers']}")
    print(f"  Arquitectura      : {' -> '.join(str(s) for s in sizes)}")
    print(f"  Épocas acumuladas : {hparams['epochs']}")
    print("  Optimizaciones:")
    print(f"    Optimizador     : {opts['optimizer']} (learning rate {opts['learning_rate']})")
    print(f"    Tamaño de lote  : {opts['batch_size']}")
    print(f"    Weight decay    : {opts['weight_decay']}")
    print(f"    Dropout         : {opts['dropout'] if opts['dropout'] > 0 else 'no'}")
    print(f"    Batch norm      : {'sí' if opts['batch_norm'] else 'no'}")
    early = f"sí (paciencia {opts['patience']})" if opts["early_stopping"] else "no"
    print(f"    Early stopping  : {early}")
    print("=" * 56)

## 4. Entrenar, probar, guardar y cargar

In [ ]:
_device_cache = {}


def device_data(classification, train):
    """Keeps each split on the training device so batches don't get copied every epoch."""
    key = (classification, train)
    if key not in _device_cache:
        images, labels = get_data(classification, train)
        _device_cache[key] = (images.to(DEVICE), labels.to(DEVICE))
    return _device_cache[key]


class Network:
    """A model plus its hyperparameters, optimizer and training history."""

    def __init__(self, hparams, state_dict=None):
        self.hparams = hparams
        self.model = build_model(hparams)
        if state_dict is not None:
            self.model.load_state_dict(state_dict)
        self.model.to(DEVICE)
        self.optimizer = None  # created on first training run

    def reset_optimizer(self):
        self.optimizer = None

    def make_optimizer(self):
        opts = self.hparams["optimizations"]
        params = self.model.parameters()
        if opts["optimizer"] == "sgd":
            return torch.optim.SGD(params, lr=opts["learning_rate"], momentum=0.9,
                                   weight_decay=opts["weight_decay"])
        return torch.optim.Adam(params, lr=opts["learning_rate"],
                                weight_decay=opts["weight_decay"])

    def set_dropout(self, probability):
        """Changes the dropout probability of the existing Dropout layers."""
        dropouts = [m for m in self.model if isinstance(m, nn.Dropout)]
        if not dropouts:
            return False
        for layer in dropouts:
            layer.p = probability
        self.hparams["optimizations"]["dropout"] = probability
        return True

    def batch_loss(self, images, labels, loss_fn, batch_size):
        """Average loss over a dataset, without gradients."""
        self.model.eval()
        total = 0.0
        with torch.no_grad():
            for start in range(0, len(labels), batch_size):
                x = images[start:start + batch_size]
                y = labels[start:start + batch_size]
                total += loss_fn(self.model(x), y).item() * len(y)
        return total / len(labels)

    def train(self, epochs):
        """Trains for up to `epochs` epochs and returns this run's train losses."""
        opts = self.hparams["optimizations"]
        if self.optimizer is None:
            self.optimizer = self.make_optimizer()
        loss_fn = nn.CrossEntropyLoss()
        images, labels = device_data(self.hparams["classification"], train=True)

        val_images = val_labels = None
        if opts["early_stopping"]:
            generator = torch.Generator().manual_seed(SEED)
            order = torch.randperm(len(labels), generator=generator).to(DEVICE)
            n_val = int(len(labels) * opts["validation_fraction"])
            val_images, val_labels = images[order[:n_val]], labels[order[:n_val]]
            images, labels = images[order[n_val:]], labels[order[n_val:]]

        batch_size = opts["batch_size"]
        best_loss, best_state, stale = float("inf"), None, 0
        run_losses = []

        for epoch in range(1, epochs + 1):
            self.model.train()
            total = 0.0
            permutation = torch.randperm(len(labels), device=DEVICE)
            for start in range(0, len(labels), batch_size):
                batch = permutation[start:start + batch_size]
                x, y = images[batch], labels[batch]
                self.optimizer.zero_grad()
                loss = loss_fn(self.model(x), y)
                loss.backward()
                self.optimizer.step()
                total += loss.item() * len(y)
            epoch_loss = total / len(labels)
            run_losses.append(epoch_loss)
            self.hparams["loss_history"].append(epoch_loss)
            self.hparams["epochs"] += 1
            message = f"Época {self.hparams['epochs']:3d} | pérdida {epoch_loss:.4f}"

            if val_labels is not None:
                val_loss = self.batch_loss(val_images, val_labels, loss_fn, batch_size)
                self.hparams["validation_loss_history"].append(val_loss)
                message += f" | pérdida validación {val_loss:.4f}"
                if val_loss < best_loss:
                    best_loss, stale = val_loss, 0
                    best_state = copy.deepcopy(self.model.state_dict())
                else:
                    stale += 1
            print(message)

            if val_labels is not None and stale >= opts["patience"]:
                print(f"Early stopping: {stale} épocas sin mejorar en validación.")
                break

        if best_state is not None:
            self.model.load_state_dict(best_state)
            print(f"Se restauraron los pesos con menor pérdida de validación ({best_loss:.4f}).")
        return run_losses

    def predict(self, images, batch_size=1024):
        self.model.eval()
        predictions = []
        with torch.no_grad():
            for start in range(0, len(images), batch_size):
                predictions.append(self.model(images[start:start + batch_size]).argmax(dim=1))
        return torch.cat(predictions)

    def test(self):
        """Evaluates on the test set and returns (confusion matrix, metrics)."""
        images, labels = device_data(self.hparams["classification"], train=False)
        predictions = self.predict(images)
        return compute_metrics(labels.cpu(), predictions.cpu(), self.hparams["output_size"])


def compute_metrics(labels, predictions, n_classes):
    """Confusion matrix (rows = real, columns = predicted), accuracy, precision and recall."""
    confusion = torch.bincount(labels * n_classes + predictions,
                               minlength=n_classes * n_classes).reshape(n_classes, n_classes)
    confusion = confusion.numpy()
    correct = np.diag(confusion).astype(float)
    predicted_totals = confusion.sum(axis=0)
    real_totals = confusion.sum(axis=1)
    precision = np.divide(correct, predicted_totals, out=np.zeros_like(correct),
                          where=predicted_totals > 0)
    recall = np.divide(correct, real_totals, out=np.zeros_like(correct), where=real_totals > 0)
    metrics = {
        "accuracy": correct.sum() / confusion.sum(),
        "precision": precision,
        "recall": recall,
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
    }
    return confusion, metrics


def print_metrics(metrics, names):
    print(f"\n{'Clase':<14}{'Precisión':>11}{'Recall':>9}")
    for name, p, r in zip(names, metrics["precision"], metrics["recall"]):
        print(f"{name:<14}{p:>11.2%}{r:>9.2%}")
    print("-" * 34)
    print(f"{'Promedio':<14}{metrics['macro_precision']:>11.2%}{metrics['macro_recall']:>9.2%}")
    print(f"\nAccuracy: {metrics['accuracy']:.2%}")


def plot_loss(hparams):
    history = hparams["loss_history"]
    if not history:
        print("La red todavía no tiene épocas entrenadas.")
        return
    epochs = range(1, len(history) + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, history, marker="o", label="Entrenamiento")
    validation = hparams["validation_loss_history"]
    if validation:
        plt.plot(range(len(history) - len(validation) + 1, len(history) + 1), validation,
                 marker="s", label="Validación")
        plt.legend()
    plt.title(f"Pérdida por época — {hparams['architecture']}, {hparams['classification']}")
    plt.xlabel("Época")
    plt.ylabel("Pérdida (cross-entropy)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_confusion(confusion, names, title):
    size = 8 if len(names) > 4 else 5
    fig, ax = plt.subplots(figsize=(size, size * 0.85))
    image = ax.imshow(confusion, cmap="Blues")
    fig.colorbar(image, ax=ax)
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticklabels(names)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_title(title)
    threshold = confusion.max() / 2
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, int(confusion[i, j]), ha="center", va="center",
                    color="white" if confusion[i, j] > threshold else "black")
    plt.tight_layout()
    plt.show()


def default_filename(hparams):
    return (f"pytorch_fashion_{NOTEBOOK_NUMBER}_{hparams['classification']}_"
            f"{hparams['architecture']}.pth")


def save_network(network, path):
    state_dict = {k: v.detach().cpu() for k, v in network.model.state_dict().items()}
    torch.save({"state_dict": state_dict, "hyperparameters": copy.deepcopy(network.hparams)}, path)


def load_network(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    hparams = checkpoint["hyperparameters"]
    options = dict(DEFAULT_OPTIMIZATIONS)
    options.update(hparams.get("optimizations", {}))
    hparams["optimizations"] = options
    hparams.setdefault("loss_history", [])
    hparams.setdefault("validation_loss_history", [])
    return Network(hparams, checkpoint["state_dict"])

## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
class ExitMenu(Exception):
    """Raised when there is no more input (closed stdin, interrupt, or a kernel without stdin)."""


def ask(prompt):
    try:
        return input(prompt).strip()
    except (EOFError, KeyboardInterrupt, RuntimeError):
        # RuntimeError covers Jupyter's StdinNotImplementedError (e.g. headless runs).
        raise ExitMenu


def ask_int(prompt, default, minimum=1):
    while True:
        text = ask(f"{prompt} [{default}]: ")
        if text == "":
            return default
        try:
            value = int(text)
            if value >= minimum:
                return value
        except ValueError:
            pass
        print(f"  Ingrese un número entero mayor o igual a {minimum}.")


def ask_float(prompt, default, minimum=0.0, maximum=None):
    while True:
        text = ask(f"{prompt} [{default}]: ").replace(",", ".")
        if text == "":
            return default
        try:
            value = float(text)
            if value >= minimum and (maximum is None or value < maximum):
                return value
        except ValueError:
            pass
        limit = f" y menor que {maximum}" if maximum is not None else ""
        print(f"  Ingrese un número mayor o igual a {minimum}{limit}.")


def ask_choice(prompt, options, default):
    """options: {key typed by the user: value}. Returns the value."""
    keys = "/".join(options)
    default_key = next(k for k, v in options.items() if v == default)
    while True:
        text = ask(f"{prompt} ({keys}) [{default_key}]: ").lower()
        if text == "":
            return default
        if text in options:
            return options[text]
        print(f"  Opción no válida. Use una de: {keys}.")


def ask_yes_no(prompt, default):
    return ask_choice(prompt, {"s": True, "n": False}, default)


def ask_path(prompt, default):
    text = ask(f"{prompt} [{default}]: ").strip('"').strip("'")
    return text or default


def create_network_from_menu():
    print("\n--- CREAR RED NUEVA ---")
    classification = ask_choice("Clasificación: o = original (10 clases), s = simplificada (4 clases)",
                                {"o": "original", "s": "simplified"}, DEFAULT_CLASSIFICATION)
    hidden_layers = ask_int("Número de capas ocultas", DEFAULT_HIDDEN_LAYERS)
    architecture = ask_choice("Tipo: r = rectangular, c = convolucional",
                              {"r": "rectangular", "c": "convolutional"}, DEFAULT_ARCHITECTURE)
    neurons = DEFAULT_NEURONS
    if architecture == "rectangular":
        neurons = ask_int("Neuronas por capa oculta", DEFAULT_NEURONS)
    print("Optimizaciones de la arquitectura (Enter = valor por defecto):")
    batch_norm = ask_yes_no("¿Usar batch normalization?", DEFAULT_OPTIMIZATIONS["batch_norm"])
    dropout = ask_float("Probabilidad de dropout (0 = sin dropout)",
                        DEFAULT_OPTIMIZATIONS["dropout"], 0.0, 1.0)
    hparams = new_hyperparameters(classification, architecture, hidden_layers, neurons,
                                  {"batch_norm": batch_norm, "dropout": dropout})
    print("\nRed creada.")
    return Network(hparams)


def load_network_from_menu():
    path = ask_path("Ruta del archivo .pth", default_filename(new_hyperparameters()))
    try:
        network = load_network(path)
        print(f"\nRed cargada desde {path}.")
        return network
    except Exception as error:
        print(f"No se pudo cargar la red: {error}")
        return None


def choose_network():
    while True:
        print("\n--- INICIO ---")
        print("1. Crear una red nueva")
        print("2. Cargar una red desde un archivo .pth")
        option = ask("Seleccione una opción (1-2): ")
        if option == "1":
            return create_network_from_menu()
        if option == "2":
            network = load_network_from_menu()
            if network is not None:
                return network
        else:
            print("Opción no válida.")


def configure_optimizations(network):
    opts = network.hparams["optimizations"]
    print("\n--- OPTIMIZACIONES (Enter = mantener el valor actual) ---")
    opts["optimizer"] = ask_choice("Optimizador", {"adam": "adam", "sgd": "sgd"}, opts["optimizer"])
    opts["learning_rate"] = ask_float("Learning rate", opts["learning_rate"], 1e-7)
    opts["weight_decay"] = ask_float("Weight decay (0 = desactivado)", opts["weight_decay"], 0.0)
    opts["batch_size"] = ask_int("Tamaño de lote", opts["batch_size"])
    opts["early_stopping"] = ask_yes_no("¿Usar early stopping?", opts["early_stopping"])
    if opts["early_stopping"]:
        opts["patience"] = ask_int("Paciencia (épocas sin mejorar)", opts["patience"])
    if opts["dropout"] > 0:
        network.set_dropout(ask_float("Probabilidad de dropout", opts["dropout"], 0.0, 1.0))
    else:
        print("Dropout y batch normalization se eligen al crear la red, porque cambian sus capas.")
    network.reset_optimizer()
    print("Optimizaciones actualizadas.")


def run_menu():
    print("TAREA 3 — CLASIFICACIÓN DE FASHION MNIST CON PYTORCH")
    try:
        network = choose_network()
        while True:
            print()
            describe(network.hparams)
            print("--- MENÚ ---")
            print("1. Entrenar")
            print("2. Probar con el conjunto de prueba")
            print("3. Guardar la red (.pth)")
            print("4. Configurar optimizaciones")
            print("5. Crear o cargar otra red")
            print("6. Salir")
            option = ask("Seleccione una opción (1-6): ")
            if option == "1":
                epochs = ask_int("Número de épocas", 10)
                network.train(epochs)
                plot_loss(network.hparams)
            elif option == "2":
                confusion, metrics = network.test()
                names = class_names(network.hparams["classification"])
                print_metrics(metrics, names)
                plot_confusion(confusion, names,
                               f"Matriz de confusión — accuracy {metrics['accuracy']:.2%}")
            elif option == "3":
                path = ask_path("Ruta donde guardar", default_filename(network.hparams))
                try:
                    save_network(network, path)
                    print(f"Red guardada en {path}.")
                except Exception as error:
                    print(f"No se pudo guardar la red: {error}")
            elif option == "4":
                configure_optimizations(network)
            elif option == "5":
                network = choose_network()
            elif option == "6":
                print("Hasta luego.")
                break
            else:
                print("Opción no válida.")
    except ExitMenu:
        print("\nNo hay más entrada; se cierra el menú.")


run_menu()